In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

df = pd.read_csv('/content/product_sales_clean.csv')
customers = pd.read_csv('/content/customer_segmentation.csv')

1. Para cada cliente, extraer su categoría principal y su gasto EN ESA CATEGORÍA

In [ ]:
customer_category_spending = df.groupby(['Customer_ID_Numeric', 'Category']).agg({
    'Revenue': 'sum',
    'Profit': 'sum',
    'Quantity': 'sum'
}).reset_index()

customer_category_spending.columns = ['Customer_ID_Numeric', 'Category', 'Revenue_In_Category', 'Profit_In_Category', 'Quantity_In_Category']

print(f"Total Customer-Category pairs: {len(customer_category_spending):,}")
print(f"\nCategorías encontradas:")
for cat in sorted(customer_category_spending['Category'].unique()):
    count = len(customer_category_spending[customer_category_spending['Category'] == cat])
    avg_rev = customer_category_spending[customer_category_spending['Category'] == cat]['Revenue_In_Category'].mean()
    print(f"  {cat:20} : {count:6,} clientes, ${avg_rev:8.2f} promedio")


Total Customer-Category pairs: 199,154

Categorías encontradas:
  Accessories          : 35,797 clientes, $  282.52 promedio
  Clothing & Apparel   : 62,003 clientes, $  437.63 promedio
  Electronics          : 50,981 clientes, $ 1127.59 promedio
  Home & Furniture     : 50,373 clientes, $  946.43 promedio


2. Mergear R_Score global

In [ ]:
customer_category_spending = customer_category_spending.merge(
    customers[['Customer_ID_Numeric', 'R_Score', 'Segment']],
    on='Customer_ID_Numeric',
    how='left'
)
print(f"Merge completo: {len(customer_category_spending)} rows")

Merge completo: 199154 rows


3. Crear M_Score POR CATEGORÍA

In [ ]:
# Lista unica y ordenada alfabeticamente de las categorias
categories = sorted(customer_category_spending['Category'].unique())

for category in categories:
    print(f"\n  {category}:")

    category_data = customer_category_spending[customer_category_spending['Category'] == category].copy()

    print(f"    Clientes: {len(category_data):,}")
    print(f"    Revenue mean: ${category_data['Revenue_In_Category'].mean():.2f}")
    print(f"    Revenue median: ${category_data['Revenue_In_Category'].median():.2f}")

    # Calcular M_Score usando qcut
    category_data['M_Score_Category'] = pd.qcut(
        category_data['Revenue_In_Category'],
        q=5,
        labels=[1, 2, 3, 4, 5],
        duplicates='drop'
    ).astype(int)

    # Actualizar en tabla principal
    customer_category_spending.loc[
        customer_category_spending['Category'] == category,
        'M_Score_Category'
    ] = category_data['M_Score_Category']

    print(f"    M_Score distribution:")
    dist = category_data['M_Score_Category'].value_counts().sort_index()
    for score, count in dist.items():
        print(f"      Score {score}: {count:,} clientes")


  Accessories:
    Clientes: 35,797
    Revenue mean: $282.52
    Revenue median: $213.66
    M_Score distribution:
      Score 1: 7,160 clientes
      Score 2: 7,159 clientes
      Score 3: 7,160 clientes
      Score 4: 7,158 clientes
      Score 5: 7,160 clientes

  Clothing & Apparel:
    Clientes: 62,003
    Revenue mean: $437.63
    Revenue median: $332.29
    M_Score distribution:
      Score 1: 12,402 clientes
      Score 2: 12,399 clientes
      Score 3: 12,401 clientes
      Score 4: 12,400 clientes
      Score 5: 12,401 clientes

  Electronics:
    Clientes: 50,981
    Revenue mean: $1127.59
    Revenue median: $839.82
    M_Score distribution:
      Score 1: 10,198 clientes
      Score 2: 10,196 clientes
      Score 3: 10,196 clientes
      Score 4: 10,195 clientes
      Score 5: 10,196 clientes

  Home & Furniture:
    Clientes: 50,373
    Revenue mean: $946.43
    Revenue median: $711.99
    M_Score distribution:
      Score 1: 10,076 clientes
      Score 2: 10,073 client

4. Crear segmentos POR CATEGORIA

In [ ]:
def segment_customer_by_category(r_score, m_score_category):
    """Asignar segmento basado en R global + M categoría"""
    if pd.isna(r_score) or pd.isna(m_score_category):
        return 'Unknown'

    r_score = int(r_score)
    m_score_category = int(m_score_category)

    if r_score >= 4 and m_score_category >= 4:
        return 'Champions'
    elif r_score >= 4 and m_score_category >= 2:
        return 'Potential'
    elif r_score <= 2 and m_score_category >= 4:
        return 'High_Value'
    elif r_score <= 2 and m_score_category <= 2:
        return 'Lost'
    elif r_score >= 3 and m_score_category >= 3:
        return 'Loyal'
    else:
        return 'At_Risk'

customer_category_spending['Segment_Category'] = customer_category_spending.apply(
    lambda row: segment_customer_by_category(row['R_Score'], row['M_Score_Category']),
    axis=1
)

print(f"Segmentos creados")

Segmentos creados


5. Análisis por Segmento y Categoría

In [ ]:

for category in categories:
    print(f"\n{category}:")
    cat_data = customer_category_spending[customer_category_spending['Category'] == category]

    segment_dist = cat_data['Segment_Category'].value_counts().sort_values(ascending=False)
    for seg, count in segment_dist.items():
        pct = (count / len(cat_data)) * 100
        avg_rev = cat_data[cat_data['Segment_Category'] == seg]['Revenue_In_Category'].mean()
        print(f"  {seg:15} : {count:6,} ({pct:5.1f}%) - ${avg_rev:8.2f} promedio")


Accessories:
  At_Risk         :  8,572 ( 23.9%) - $  141.16 promedio
  Lost            :  5,818 ( 16.3%) - $  121.60 promedio
  High_Value      :  5,801 ( 16.2%) - $  478.32 promedio
  Potential       :  5,635 ( 15.7%) - $  184.15 promedio
  Champions       :  5,600 ( 15.6%) - $  478.36 promedio
  Loyal           :  4,371 ( 12.2%) - $  389.95 promedio

Clothing & Apparel:
  At_Risk         : 14,783 ( 23.8%) - $  189.91 promedio
  Champions       : 10,096 ( 16.3%) - $  780.48 promedio
  Potential       : 10,086 ( 16.3%) - $  272.56 promedio
  Lost            :  9,894 ( 16.0%) - $  149.05 promedio
  High_Value      :  9,718 ( 15.7%) - $  775.72 promedio
  Loyal           :  7,426 ( 12.0%) - $  630.89 promedio

Electronics:
  At_Risk         : 12,217 ( 24.0%) - $  514.87 promedio
  Champions       :  8,300 ( 16.3%) - $ 1978.76 promedio
  Potential       :  8,234 ( 16.2%) - $  705.59 promedio
  Lost            :  8,081 ( 15.9%) - $  421.94 promedio
  High_Value      :  8,062 ( 15.8%) - $

6. Comparación dentro de cada categoría

In [ ]:
for category in categories:
    cat_data = customer_category_spending[customer_category_spending['Category'] == category]

    champions = cat_data[cat_data['Segment_Category'] == 'Champions']['Revenue_In_Category']
    potential = cat_data[cat_data['Segment_Category'] == 'Potential']['Revenue_In_Category']
    high_value = cat_data[cat_data['Segment_Category'] == 'High_Value']['Revenue_In_Category']
    lost = cat_data[cat_data['Segment_Category'] == 'Lost']['Revenue_In_Category']
    loyal = cat_data[cat_data['Segment_Category'] == 'Loyal']['Revenue_In_Category']
    at_risk = cat_data[cat_data['Segment_Category'] == 'At_Risk']['Revenue_In_Category']

    print(f"\n{category}:")

    if len(champions) > 0:
        print(f"  Champions:  {len(champions):6,} clientes | ${champions.mean():8.2f} promedio")
    if len(potential) > 0:
        print(f"  Potential:  {len(potential):6,} clientes | ${potential.mean():8.2f} promedio")
        if len(champions) > 0 and potential.mean() > 0:
            ratio = champions.mean() / potential.mean()
            print(f"    Champions es {ratio:.2f}x mayor que Potential")
    if len(loyal) > 0:
        print(f"  Loyal:      {len(loyal):6,} clientes | ${loyal.mean():8.2f} promedio")
    if len(high_value) > 0:
        print(f"  High_Value: {len(high_value):6,} clientes | ${high_value.mean():8.2f} promedio")
    if len(at_risk) > 0:
        print(f"  At_Risk:    {len(at_risk):6,} clientes | ${at_risk.mean():8.2f} promedio")
    if len(lost) > 0:
        print(f"  Lost:       {len(lost):6,} clientes | ${lost.mean():8.2f} promedio")


Accessories:
  Champions:   5,600 clientes | $  478.36 promedio
  Potential:   5,635 clientes | $  184.15 promedio
    Champions es 2.60x mayor que Potential
  Loyal:       4,371 clientes | $  389.95 promedio
  High_Value:  5,801 clientes | $  478.32 promedio
  At_Risk:     8,572 clientes | $  141.16 promedio
  Lost:        5,818 clientes | $  121.60 promedio

Clothing & Apparel:
  Champions:  10,096 clientes | $  780.48 promedio
  Potential:  10,086 clientes | $  272.56 promedio
    Champions es 2.86x mayor que Potential
  Loyal:       7,426 clientes | $  630.89 promedio
  High_Value:  9,718 clientes | $  775.72 promedio
  At_Risk:    14,783 clientes | $  189.91 promedio
  Lost:        9,894 clientes | $  149.05 promedio

Electronics:
  Champions:   8,300 clientes | $ 1978.76 promedio
  Potential:   8,234 clientes | $  705.59 promedio
    Champions es 2.80x mayor que Potential
  Loyal:       6,087 clientes | $ 1598.24 promedio
  High_Value:  8,062 clientes | $ 1962.76 promedio
  At_R

7. Guardado

In [ ]:
customer_category_spending.to_csv('customer_segmentation_BY_CATEGORY.csv', index=False)
print("Guardado: customer_segmentation_BY_CATEGORY.csv")

# Resumen
segment_category_summary = customer_category_spending.groupby(['Category', 'Segment_Category']).agg({
    'Customer_ID_Numeric': 'count',
    'Revenue_In_Category': ['sum', 'mean'],
    'Profit_In_Category': ['mean']
}).round(2)
segment_category_summary.columns = ['Num_Customers', 'Total_Revenue', 'Avg_Revenue', 'Avg_Profit']
segment_category_summary.to_csv('segment_category_summary.csv')
print("Guardado: segment_category_summary.csv")

8. Visualizar diferencias

In [ ]:
print(f"\nTotal Customer-Category combinations: {len(customer_category_spending):,}")
print(f"Categorías analizadas: {customer_category_spending['Category'].nunique()}")
print(f"Clientes únicos: {customer_category_spending['Customer_ID_Numeric'].nunique():,}")
print(f"Revenue total: ${customer_category_spending['Revenue_In_Category'].sum():,.2f}")

print("\n Análisis completado exitosamente")


Total Customer-Category combinations: 199,154
Categorías analizadas: 4
Clientes únicos: 197,000
Revenue total: $142,407,744.93

 Análisis completado exitosamente
